# Parte Comum: K-Means, Passada 2 (k fixo)

Dataset: Obesity (UCI 544), arquivo `data/parte_comum/obesity.csv`.

## Etapa 1: Padronização das variáveis

In [ ]:
# [IA] Claude Code – importacao das bibliotecas
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [ ]:
# [IA] Claude Code – leitura do dataset (caminho relativo a pasta do notebook)
CAMINHO_DADOS = Path("../../data/parte_comum/obesity.csv")

df = pd.read_csv(CAMINHO_DADOS)
print(df.shape)
df.head()

In [ ]:
# [IA] Claude Code – separar variaveis de entrada (X) e classe real (y)
# A classe real NAO entra no K-Means; fica guardada para o mapeamento e as metricas.
COLUNA_CLASSE = "NObeyesdad"

y = df[COLUNA_CLASSE].copy()
X = df.drop(columns=[COLUNA_CLASSE]).copy()

print("Classes reais:", y.nunique())
print(y.value_counts())

In [ ]:
# [IA] Claude Code – variaveis numericas usadas como entrada do K-Means (lista explicita)
# O K-Means agrupa por distancia euclidiana, que so faz sentido para variaveis numericas.
# As categoricas (Gender, family_history_with_overweight, FAVC, CAEC, SMOKE, SCC, CALC, MTRANS)
# continuam sendo analisadas na EDA; so nao entram no agrupamento.
# A Passada 1 (kmeans_passada1.ipynb) deve usar exatamente esta mesma lista.
VARIAVEIS_KMEANS = ["Age", "Height", "Weight", "FCVC", "NCP", "CH2O", "FAF", "TUE"]

X = X[VARIAVEIS_KMEANS]

# Conferencia: todas as variaveis escolhidas sao numericas
assert all(pd.api.types.is_numeric_dtype(X[c]) for c in VARIAVEIS_KMEANS), "Alguma variavel nao e numerica"
X.head()

**Escolha das variáveis (decisão do grupo):** Escolhemos as oito variáveis numéricas porque o K-Means utiliza cálculos de distância para formar os grupos. As variáveis representam características físicas, idade e hábitos relacionados à alimentação, consumo de água e atividade física. As variáveis categóricas continuam sendo analisadas na EDA, mas não entram no agrupamento.


In [ ]:
# [IA] Claude Code – padronizacao (z-score): media 0 e desvio padrao 1 em cada variavel
scaler = StandardScaler()
X_padronizado = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index,
)
X_padronizado.head()

In [ ]:
# [IA] Claude Code – conferir a padronizacao (media ~ 0 e desvio ~ 1)
conferencia = pd.DataFrame({
    "media_antes": X.mean(),
    "desvio_antes": X.std(ddof=0),
    "media_depois": X_padronizado.mean().round(6),
    "desvio_depois": X_padronizado.std(ddof=0).round(6),
})
conferencia

### Por que padronizar antes do K-Means?

A padronização é importante no K-Means. Esse algoritmo utiliza a distância euclidiana para calcular a proximidade entre os dados e formar os grupos. Como nossas variáveis possuem escalas diferentes, como altura, peso e idade, sem a padronização algumas variáveis poderiam ter muito mais influência no cálculo das distâncias do que outras. Por isso utilizamos o StandardScaler, que deixa as variáveis com média próxima de zero e desvio padrão igual a um, fazendo com que todas tenham uma participação mais equilibrada na formação dos grupos.


## Etapa 2: K-Means com k = número de classes reais

In [ ]:
# [IA] Claude Code – k fixo no numero de classes reais do dataset
from sklearn.cluster import KMeans

K_FIXO = y.nunique()
print("k fixo =", K_FIXO)

In [ ]:
# [IA] Claude Code – executar o K-Means sobre os dados padronizados
# random_state fixo para o resultado ser reproduzivel; n_init=10 roda 10 inicializacoes e fica com a melhor
kmeans_fixo = KMeans(n_clusters=K_FIXO, n_init=10, random_state=42)
clusters_fixo = kmeans_fixo.fit_predict(X_padronizado)

print("Inercia (soma das distancias ao quadrado):", round(kmeans_fixo.inertia_, 2))

In [ ]:
# [IA] Claude Code – guardar o cluster de cada linha junto com a classe real (usado no mapeamento e nas metricas)
resultado_fixo = pd.DataFrame({
    "cluster": clusters_fixo,
    "classe_real": y.values,
}, index=X.index)

# Quantidade de pontos em cada cluster
resultado_fixo["cluster"].value_counts().sort_index().rename("qtd_pontos").to_frame()

In [ ]:
# [IA] Claude Code – centroides de cada cluster na escala original das variaveis
centroides_fixo = pd.DataFrame(
    scaler.inverse_transform(kmeans_fixo.cluster_centers_),
    columns=X.columns,
).round(2)
centroides_fixo.index.name = "cluster"
centroides_fixo

## Etapa 3: Visualização dos grupos

In [ ]:
# [IA] Claude Code – reduzir as 8 variaveis padronizadas para 2 componentes (PCA) so para desenhar
# O K-Means continua rodando nas 8 variaveis; o PCA serve apenas para projetar os pontos no plano.
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_padronizado)
centroides_2d = pca.transform(pd.DataFrame(kmeans_fixo.cluster_centers_, columns=X_padronizado.columns))

var_explicada = pca.explained_variance_ratio_ * 100
print(f"Variancia explicada: PC1 = {var_explicada[0]:.1f}%, PC2 = {var_explicada[1]:.1f}%, "
      f"total = {var_explicada.sum():.1f}%")

In [ ]:
# [IA] Claude Code – grafico lado a lado: clusters do K-Means x classes reais, na mesma projecao PCA
ordem_classes = [
    "Insufficient_Weight", "Normal_Weight",
    "Overweight_Level_I", "Overweight_Level_II",
    "Obesity_Type_I", "Obesity_Type_II", "Obesity_Type_III",
]
cores = plt.get_cmap("tab10").colors

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6), sharex=True, sharey=True)

for i, cluster in enumerate(sorted(resultado_fixo["cluster"].unique())):
    filtro = resultado_fixo["cluster"].values == cluster
    ax1.scatter(coords[filtro, 0], coords[filtro, 1], s=12, alpha=0.6,
                color=cores[i], label=f"Cluster {cluster}")
ax1.scatter(centroides_2d[:, 0], centroides_2d[:, 1], marker="X", s=200,
            color="black", edgecolor="white", linewidth=1.5, label="Centroide")
ax1.set_title(f"Passada 2: clusters do K-Means (k = {K_FIXO})")

for i, classe in enumerate(ordem_classes):
    filtro = y.values == classe
    ax2.scatter(coords[filtro, 0], coords[filtro, 1], s=12, alpha=0.6,
                color=cores[i], label=classe)
ax2.set_title("Classes reais (NObeyesdad)")

for ax in (ax1, ax2):
    ax.set_xlabel(f"Componente principal 1 ({var_explicada[0]:.1f}% da variancia)")
    ax.legend(fontsize=8, markerscale=1.5, loc="best")
ax1.set_ylabel(f"Componente principal 2 ({var_explicada[1]:.1f}% da variancia)")

fig.suptitle("Obesity: grupos encontrados x classes reais (projecao PCA das 8 variaveis padronizadas)")
fig.tight_layout()

PASTA_SAIDA = Path("../../out/parte_comum")
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
fig.savefig(PASTA_SAIDA / "passada2_clusters_pca.png", dpi=150, bbox_inches="tight")
plt.show()

**Por que esta visualização (decisão do grupo):** Escolhemos utilizar a dispersão dos componentes do PCA porque ela permite visualizar os grupos formados pelo K-Means em duas dimensões. Também decidimos manter a comparação com as classes reais para facilitar a identificação das semelhanças e diferenças entre os agrupamentos.


In [ ]:
# [IA] Claude Code – quanto cada variavel pesa em cada componente (ajuda a ler os eixos do grafico)
pd.DataFrame(
    pca.components_.T,
    index=VARIAVEIS_KMEANS,
    columns=["PC1", "PC2"],
).round(3)

## Etapa 4: Mapeamento cluster → classe majoritária

In [ ]:
# [IA] Claude Code – tabela de contagem: quantas pessoas de cada classe real caem em cada cluster
contagem_cluster_classe = pd.crosstab(
    resultado_fixo["cluster"],
    resultado_fixo["classe_real"],
)[ordem_classes]
contagem_cluster_classe

In [ ]:
# [IA] Claude Code – cada cluster recebe o nome da classe real que mais aparece dentro dele
mapa_cluster_classe = contagem_cluster_classe.idxmax(axis=1)

resumo_mapeamento = pd.DataFrame({
    "classe_majoritaria": mapa_cluster_classe,
    "qtd_classe_majoritaria": contagem_cluster_classe.max(axis=1),
    "qtd_total_cluster": contagem_cluster_classe.sum(axis=1),
})
resumo_mapeamento["pct_classe_majoritaria"] = (
    resumo_mapeamento["qtd_classe_majoritaria"] / resumo_mapeamento["qtd_total_cluster"] * 100
).round(1)
resumo_mapeamento

In [ ]:
# [IA] Claude Code – aplicar o mapeamento a cada linha (rotulo previsto usado na matriz de confusao)
resultado_fixo["classe_mapeada"] = resultado_fixo["cluster"].map(mapa_cluster_classe)

# Conferencia: classes reais que receberam mais de um cluster e classes que nao receberam nenhum
clusters_por_classe = mapa_cluster_classe.value_counts()
print("Clusters por classe mapeada:")
print(clusters_por_classe)
print("Classes reais sem nenhum cluster:", sorted(set(ordem_classes) - set(mapa_cluster_classe)))

resultado_fixo.head()

## Etapa 5: Matriz de confusão e métricas

Linhas = classe real; colunas = classe mapeada a partir do cluster.

In [ ]:
# [IA] Claude Code – matriz de confusao (quadrada: k = numero de classes)
from sklearn.metrics import confusion_matrix

matriz_fixo = pd.DataFrame(
    confusion_matrix(resultado_fixo["classe_real"], resultado_fixo["classe_mapeada"], labels=ordem_classes),
    index=pd.Index(ordem_classes, name="real"),
    columns=pd.Index(ordem_classes, name="mapeada"),
)
matriz_fixo

In [ ]:
# [IA] Claude Code – heatmap da matriz de confusao em % por linha (cada classe real soma 100%), com a contagem em cada celula
import seaborn as sns

pct_linha = matriz_fixo.div(matriz_fixo.sum(axis=1), axis=0) * 100
rotulos = pct_linha.round(1).astype(str).replace(r"\.", ",", regex=True) + "%\n(" + matriz_fixo.astype(str) + ")"

fig, ax = plt.subplots(figsize=(10, 7.5))
sns.heatmap(pct_linha, annot=rotulos, fmt="", cmap="Blues", vmin=0, vmax=100,
            cbar_kws={"label": "% das pessoas da classe real (cada linha soma 100%)"}, ax=ax)
ax.set_title(f"Passada 2 (k = {K_FIXO}): matriz de confusao em % por linha\n(cada linha e uma classe real; entre parenteses, a quantidade de pessoas)")
ax.set_xlabel("Classe mapeada (classe majoritaria do cluster)")
ax.set_ylabel("Classe real (NObeyesdad)")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
fig.tight_layout()
fig.savefig(PASTA_SAIDA / "passada2_matriz_confusao.png", dpi=150, bbox_inches="tight")
plt.show()


**Por que esta visualização (decisão do grupo):** Escolhemos o mapa de calor com porcentagens por linha porque ele facilita a comparação entre as sete classes, mesmo quando possuem quantidades diferentes de pessoas. Também mantivemos as contagens absolutas nas células para não perder os valores reais.


In [ ]:
# [IA] Claude Code – acuracia, precisao e recall calculados a partir da matriz
m = matriz_fixo.values
acertos = np.diag(m)                     # diagonal: real == mapeada
total_real = m.sum(axis=1)               # soma da linha: pessoas de cada classe real
total_mapeado = m.sum(axis=0)            # soma da coluna: pessoas rotuladas com cada classe

acuracia_fixo = acertos.sum() / m.sum()

# Precisao = acertos / total rotulado com a classe. Se nenhuma pessoa recebeu a classe
# (nenhum cluster mapeado para ela), a divisao e 0/0: registramos 0 e marcamos na tabela.
precisao = np.divide(acertos, total_mapeado, out=np.zeros(len(acertos)), where=total_mapeado > 0)
recall = acertos / total_real

metricas_fixo = pd.DataFrame({
    "precisao": precisao.round(3),
    "recall": recall.round(3),
    "qtd_real": total_real,
    "qtd_mapeada": total_mapeado,
    "sem_cluster_mapeado": total_mapeado == 0,
}, index=pd.Index(ordem_classes, name="classe"))

print(f"Acuracia (Passada 2): {acuracia_fixo:.3f}")
metricas_fixo

In [ ]:
# [IA] Claude Code – conferencia com as funcoes do scikit-learn (devem dar os mesmos valores)
from sklearn.metrics import accuracy_score, precision_score, recall_score

args = dict(y_true=resultado_fixo["classe_real"], y_pred=resultado_fixo["classe_mapeada"], labels=ordem_classes)
assert np.isclose(accuracy_score(resultado_fixo["classe_real"], resultado_fixo["classe_mapeada"]), acuracia_fixo)
assert np.allclose(precision_score(average=None, zero_division=0, **args), precisao)
assert np.allclose(recall_score(average=None, zero_division=0, **args), recall)
print("Metricas conferem com o scikit-learn.")